### MODEL1

ParkWise Model 1: Overhead Car Counting via Transfer Learning (ResNet-18)

### Description:
Trains a continuous scalar regression model on the COWC overhead dataset to count cars in aerial image patches using CSV based labels. Evaluates inference over Nairobi parking lots via a sliding window and updates `nairobi_parking_spotcheck.csv`.

In [1]:
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision.transforms as T
from torchvision.models import ResNet18_Weights, resnet18
from tqdm import tqdm

Imports

What it does: Pulls in every library the pipeline needs: pathlib for file paths, numpy and pandas for data handling, PIL for opening images, torch/torchvision for building and training the neural network, and tqdm for progress bars during training.

In [3]:
#CONFIGURATION
class ParkWiseConfig:
    """
    Holds all path, device, and hyperparameter configuration for Model 1.

    This is a direct, unmodified transcription of the original script's
    "1. SETUP & PATH CONFIGURATION" section — every value is identical.
    Grouping these into a class lets every other class in this file take
    a single `config` object instead of relying on module-level globals,
    without changing what any of the values actually are.
    """

    def __init__(self):
        # --- Paths (identical to original) ---
        self.PROJECT_DIR = Path("/home/nia/Downloads/parkwise")

        self.DATASET_PATCHES_DIR = Path("/home/nia/Downloads/parkwise/DetectionPatches_512x512_ALL")
        self.CSV_PATH = self.DATASET_PATCHES_DIR / "object_count.csv"

        self.SPOTCHECK_PATH = self.PROJECT_DIR / "parkwise_final_maybe/nairobi_parking_spotcheck.csv"
        self.NAIROBI_IMAGERY_DIR = self.PROJECT_DIR / "Images"

        self.MODEL_SAVE_PATH = self.PROJECT_DIR / "model1_artifacts/parkwise_model1_resnet18.pt"
        self.MODEL_SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

        # --- Device ---
        self.DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using compute device: {self.DEVICE}")

        # --- Dataset parameters (identical to original) ---
        self.IMAGE_SIZE = (224, 224)
        self.PATCH_STRIDE = 160  # Stride for sliding window inference on Nairobi images
        self.BATCH_SIZE = 32
        self.EPOCHS = 15
        self.LEARNING_RATE = 1e-4

        # Location to hold out for spatial test splitting
        self.HELD_OUT_LOCATION = "Potsdam"

What it does: Defines every path (where the dataset lives, where the model gets saved, where Nairobi imagery is), picks the compute device (GPU if available, otherwise CPU), and sets the hyperparameters which are image size, batch size, epochs, learning rate, which location to hold out for testing.

Produces a single config object that every other class reads from for centralizing these values in one place.

In [4]:
#COLUMN INSPECTION & DATASET LOADER
class DatasetInspector:
    """
    Wraps the original `setup_target_and_filename_cols` function.

    Responsible for looking at the raw object_count.csv columns and
    deciding (a) which column(s) represent the vehicle count target,
    and (b) which columns hold the filename / folder for each row.
    Logic is unchanged from the original script.
    """

    @staticmethod
    def setup_target_and_filename_cols(df):
        """
        Creates a 'Total_Car_Count' column by summing all non-negative vehicle columns.
        (Unmodified from original script.)
        """
        print("\n--- CSV Structure Inspection ---")
        print(f"Columns available: {list(df.columns)}")
        print("Sample Row 0:")
        for col in df.columns:
            print(f"  - {col}: {df[col].iloc[0]}")
        print("--------------------------------\n")

        # Define vehicle columns to sum for total car target
        vehicle_cols = [c for c in df.columns if
                        any(k in str(c).lower() for k in ["sedan", "pickup", "other", "unknown", "car", "pos"])]

        if vehicle_cols:
            df["Total_Car_Count"] = df[vehicle_cols].sum(axis=1)
            count_col = "Total_Car_Count"
            print(f"✓ Summed vehicle columns {vehicle_cols} -> Target Column: '{count_col}'")
        else:
            count_col = df.columns[2]

        filename_col = "File_Name" if "File_Name" in df.columns else df.columns[1]
        folder_col = "Folder_Name" if "Folder_Name" in df.columns else None

        print(f"Mapped Filename Column: '{filename_col}' | Folder Column: '{folder_col}'")
        return df, filename_col, count_col, folder_col


class COWCCountingCSVDataset(Dataset):
    """
    COWC Dataset class reading image patches guided by object_count.csv.
    Handles relative folder paths, extension variations, and recursive search.

    (Unmodified from original script — already a class there; kept verbatim
    including all internal logic, path resolution fallbacks, and error handling.)
    """

    def __init__(self, df, patches_dir, filename_col, count_col, folder_col=None, transform=None):
        self.patches_dir = Path(patches_dir)
        self.transform = transform
        self.labels = []
        self.valid_paths = []

        print("Building file map from dataset directory...")
        file_map = {p.name.lower(): p for p in self.patches_dir.rglob("*") if p.is_file()}
        print(f"✓ Indexed {len(file_map)} total image files under {self.patches_dir.name}")

        for _, row in df.iterrows():
            raw_filename = str(row[filename_col]).strip()

            if folder_col and folder_col in row and pd.notna(row[folder_col]):
                rel_path_str = f"{str(row[folder_col]).strip()}/{raw_filename}"
            else:
                rel_path_str = raw_filename

            direct_path = self.patches_dir / rel_path_str
            resolved_path = None

            if direct_path.is_file():
                resolved_path = direct_path
            else:
                for ext in [".png", ".jpg", ".jpeg"]:
                    if Path(f"{direct_path}{ext}").is_file():
                        resolved_path = Path(f"{direct_path}{ext}")
                        break

                if resolved_path is None:
                    base_name = Path(raw_filename).name.lower()
                    if base_name in file_map:
                        resolved_path = file_map[base_name]
                    else:
                        for ext in [".png", ".jpg", ".jpeg"]:
                            if f"{base_name}{ext}" in file_map:
                                resolved_path = file_map[f"{base_name}{ext}"]
                                break

            if resolved_path and resolved_path.is_file():
                try:
                    count_val = float(row[count_col])
                    self.valid_paths.append(resolved_path)
                    self.labels.append(count_val)
                except (ValueError, TypeError):
                    continue

    def __len__(self):
        return len(self.valid_paths)

    def __getitem__(self, idx):
        img_path = self.valid_paths[idx]
        image = Image.open(img_path).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)


class TransformFactory:
    """
    Wraps the original `get_transforms` function.
    Produces the train-time (augmented) and validation-time (plain) torchvision
    transform pipelines. Values and ordering are unchanged from the original script.
    """

    @staticmethod
    def get_transforms(image_size):
        train_transform = T.Compose([
            T.Resize(image_size),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.ColorJitter(brightness=0.2, contrast=0.2),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        val_transform = T.Compose([
            T.Resize(image_size),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        return train_transform, val_transform

### DatasetInspector, COWCCountingCSVDataset, TransformFactory

DatasetInspector.setup target and filename cols What it does: Looks at the raw object_count.csv columns, figures out which columns represent vehicle counts (sedan, pickup, unknown, etc.), sums them into a single Total_Car_Count target, and identifies which column holds the filename and which holds the folder. Hence, produces a dataframe with a new Total_Car_Count column, plus the names of the count columns to use downstream. Why it's necessary: The COWC CSV doesn't ship with one clean car count column, it has separate columns per vehicle type. The model needs a single number to predict, so this step builds that number and confirms exactly which columns to trust before any training happens.

COWCCountingCSVDataset What it does: For every row in the CSV, finds the matching image file on disk (handling folder prefixes, missing file extensions, and case differences), and stores the (image path, car count) pairs. When PyTorch asks for an item, it opens the image, applies transforms, and returns it with its label. What it produces: A PyTorch Dataset object the standard structure PyTorch's training loop expects to pull batches from. Why it's necessary: The COWC dataset is messy (500k+ files across folders, inconsistent naming). This class is the translation layer between "a row in a CSV" and "an actual image tensor the model can train on." Without it, none of the file-matching edge cases would be handled and training would crash or silently skip data.

TransformFactory.get_transforms What it does: Builds two image-processing pipelines: one for training (resize, random flips, color jitter, then normalize) and one for validation/testing (resize and normalize only, no randomness). What it produces: Two torchvision.transforms.Compose objects train_transform and val_transform. Why it's necessary: Random flips/color jitter during training act as data augmentation, which helps the model generalize instead of memorizing exact pixel patterns. But you never want randomness during evaluation, because you need consistent, repeatable predictions to measure accuracy, hence two separate pipelines.

In [5]:
#ARCHITECTURE SETUP FOR RESNET 18 REGRESSOR
class RegressorModelBuilder:
    """
    Wraps the original `build_resnet18_regressor` function.
    Builds an ImageNet-pretrained ResNet-18 backbone with its final fully-connected
    layer replaced by a small regression head (Linear -> ReLU -> Dropout -> Linear(1)).
    Unmodified from the original script.
    """

    @staticmethod
    def build_resnet18_regressor():
        model = resnet18(weights=ResNet18_Weights.DEFAULT)
        in_features = model.fc.in_features

        model.fc = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
        return model

RegressorModelBuilder

What it does: Loads a ResNet-18 architecture pretrained on ImageNet, then replaces its final classification layer with a small regression head (Linear → ReLU → Dropout → Linear(1)), which is the final laer of the Resnet that uses extracted image features to predict number of vehicles that outputs a single number instead of a class label. It produces a torch.nn.Module CNN ready to be trained. Why it's necessary: ResNet-18 pretrained on ImageNet already knows how to recognize shapes, edges, and textures which is transfer learning, and it means the model needs far less training data to become useful than training from scratch. Swapping the final layer turns it from "which of 1000 classes is this" into "how many cars are in this patch," which is the actual task here.

In [6]:
#SLIDING WINDOW INFERENCE FOR NAIROBI LOTS
class SlidingWindowCounter:
    """
    Wraps the original `count_cars_in_large_image` function.
    Slides a patch-sized window across a full-resolution Nairobi parking lot image,
    runs each patch through the trained regressor, clips negative predictions to
    zero, and sums the result into a single total car count for the whole image.
    Unmodified from the original script (including the trailing-batch handling).
    """

    @staticmethod
    def count_cars_in_large_image(model, image_path, transform, device, image_size, stride, batch_size):
        full_img = Image.open(image_path).convert("RGB")
        width, height = full_img.size

        patch_w, patch_h = image_size

        total_predicted_cars = 0.0
        patches_batch = []

        for y in range(0, height - patch_h + 1, stride):
            for x in range(0, width - patch_w + 1, stride):
                box = (x, y, x + patch_w, y + patch_h)
                patch = full_img.crop(box)
                patch_tensor = transform(patch)
                patches_batch.append(patch_tensor)

                if len(patches_batch) == batch_size:
                    batch_tensor = torch.stack(patches_batch).to(device)
                    with torch.no_grad():
                        preds = model(batch_tensor).squeeze(-1).cpu().numpy()
                        total_predicted_cars += np.sum(np.clip(preds, 0, None))
                    patches_batch = []

        if patches_batch:
            batch_tensor = torch.stack(patches_batch).to(device)
            with torch.no_grad():
                preds = model(batch_tensor).squeeze(-1).cpu().numpy()
                total_predicted_cars += np.sum(np.clip(preds, 0, None))

        return total_predicted_cars

SlidingWindowCounter

What it does: Takes one large Nairobi parking lot photo, cuts it into overlapping smaller patches to match the size the model was trained on, runs each patch through the trained model, clips any negative predictions to zero because we can't have negative cars, and adds up the predicted counts across all patches. What it produces: A single total predicted car count for the whole parking lot image. Why it's necessary: The model was only ever trained to count cars in small fixed-size patches (224×224), not full-size aerial photos. This class is what lets you apply that patch-level model to a full-resolution Nairobi parking lot photo and get one meaningful number out the other end.

In [7]:
#MAIN TRAINING & INFERENCE PIPELINE
class ParkWiseModel1Pipeline:
    """
    Wraps the original `main()` function as a class.

    Orchestrates the full Model 1 workflow in the same order as the original
    script: dataset loading -> column inspection -> location-based train/test
    split -> dataset/dataloader construction with class-imbalance sampling ->
    model build -> training loop with per-epoch MAE evaluation and best-checkpoint
    saving -> sliding-window inference over Nairobi imagery -> spotcheck CSV update.

    Every computation inside `run()` is identical to the body of the original
    `main()` function; only the surrounding structure (config object, delegation
    to the helper classes above, and storing intermediate state on `self`) differs.
    """

    def __init__(self, config: ParkWiseConfig):
        self.config = config
        # State populated as the pipeline runs (mirrors local variables in original main())
        self.model = None
        self.best_mae = float("inf")
        self.train_dataset = None
        self.test_dataset = None
        self.val_tf = None

    def run(self):
        """Runs the entire Model 1 pipeline end-to-end, unchanged from original main()."""
        cfg = self.config

        print("=" * 70)
        print("MODEL 1: CNN OVERHEAD CAR COUNTING PIPELINE")
        print("=" * 70)

        if not cfg.DATASET_PATCHES_DIR.exists():
            print(f"Directory {cfg.DATASET_PATCHES_DIR} not found.")
            return

        if not cfg.CSV_PATH.exists():
            print(f"CSV file missing at {cfg.CSV_PATH}.")
            return

        df_labels = pd.read_csv(cfg.CSV_PATH)
        print(f"✓ Loaded {len(df_labels)} records from {cfg.CSV_PATH.name}.")

        df_labels, filename_col, count_col, folder_col = DatasetInspector.setup_target_and_filename_cols(df_labels)

        search_col = folder_col if folder_col else filename_col
        held_out_mask = df_labels[search_col].astype(str).str.contains(cfg.HELD_OUT_LOCATION, case=False, na=False)

        if held_out_mask.sum() > 0:
            train_df = df_labels[~held_out_mask].reset_index(drop=True)
            test_df = df_labels[held_out_mask].reset_index(drop=True)
            print(f"✓ Split by location '{cfg.HELD_OUT_LOCATION}' -> Train: {len(train_df)} | Test: {len(test_df)}")
        else:
            train_df = df_labels.sample(frac=0.8, random_state=42)
            test_df = df_labels.drop(train_df.index).reset_index(drop=True)
            train_df = train_df.reset_index(drop=True)
            print(f"✓ Applied 80/20 random split -> Train: {len(train_df)} | Test: {len(test_df)}")

        train_tf, val_tf = TransformFactory.get_transforms(cfg.IMAGE_SIZE)
        self.val_tf = val_tf

        train_dataset = COWCCountingCSVDataset(
            train_df, cfg.DATASET_PATCHES_DIR, filename_col, count_col, folder_col=folder_col, transform=train_tf
        )
        test_dataset = COWCCountingCSVDataset(
            test_df, cfg.DATASET_PATCHES_DIR, filename_col, count_col, folder_col=folder_col, transform=val_tf
        )
        self.train_dataset = train_dataset
        self.test_dataset = test_dataset

        print(f"Active resolved images -> Train: {len(train_dataset)} | Test: {len(test_dataset)}")

        if len(train_dataset) == 0:
            print("No matching patch images found in directory! Check CSV vs folder structure.")
            return

        train_labels = np.array(train_dataset.labels)
        zero_mask = (train_labels == 0)
        num_zeros = np.sum(zero_mask)
        num_nonzeros = len(train_labels) - num_zeros

        print(f"Zero-car patches: {num_zeros} ({num_zeros / len(train_labels) * 100:.1f}%)")
        print(f"Non-zero patches: {num_nonzeros} ({num_nonzeros / len(train_labels) * 100:.1f}%)")

        weight_zero = 1.0 / max(num_zeros, 1)
        weight_nonzero = 1.0 / max(num_nonzeros, 1)
        sample_weights = np.where(zero_mask, weight_zero, weight_nonzero)

        sampler = WeightedRandomSampler(
            weights=torch.tensor(sample_weights, dtype=torch.double),
            num_samples=len(sample_weights),
            replacement=True
        )

        train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, sampler=sampler, num_workers=2)
        test_loader = DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

        model = RegressorModelBuilder.build_resnet18_regressor().to(cfg.DEVICE)
        self.model = model
        criterion = nn.L1Loss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=1e-4)

        best_mae = float("inf")

        print("\n" + "=" * 70)
        print("STARTING TRAINING")
        print("=" * 70)

        for epoch in range(1, cfg.EPOCHS + 1):
            model.train()
            running_loss = 0.0

            for images, targets in tqdm(train_loader, desc=f"Epoch {epoch}/{cfg.EPOCHS}"):
                images = images.to(cfg.DEVICE)
                targets = targets.to(cfg.DEVICE)

                optimizer.zero_grad()
                outputs = model(images).squeeze(-1)
                loss = criterion(outputs, targets)
                loss.backward()
                optimizer.step()

                running_loss += loss.item() * images.size(0)

            epoch_train_mae = running_loss / len(train_dataset)

            model.eval()
            test_abs_error = 0.0

            with torch.no_grad():
                for images, targets in test_loader:
                    images = images.to(cfg.DEVICE)
                    targets = targets.to(cfg.DEVICE)
                    outputs = model(images).squeeze(-1)
                    outputs = torch.clamp(outputs, min=0.0)
                    test_abs_error += torch.sum(torch.abs(outputs - targets)).item()

            epoch_test_mae = test_abs_error / max(len(test_dataset), 1)
            print(f"Epoch {epoch:02d} | Train MAE: {epoch_train_mae:.3f} | Test MAE: {epoch_test_mae:.3f}")

            if epoch_test_mae < best_mae:
                best_mae = epoch_test_mae
                torch.save(model.state_dict(), cfg.MODEL_SAVE_PATH)
                print(f"  ✓ Saved best model checkpoint to {cfg.MODEL_SAVE_PATH} (MAE: {best_mae:.3f})")

        self.best_mae = best_mae
        print(f"\nTraining Complete. Target Metric (MAE <= 2.5): Final Best MAE = {best_mae:.3f}")

        # ----------------------------------------------------
        # STEP 7: INFERENCE ON NAIROBI SATELLITE IMAGERY
        # ----------------------------------------------------
        print("\n" + "=" * 70)
        print("STEP 7: RUNNING INFERENCE ON NAIROBI PARKING LOTS")
        print("=" * 70)

        if cfg.MODEL_SAVE_PATH.exists():
            model.load_state_dict(torch.load(cfg.MODEL_SAVE_PATH, map_location=cfg.DEVICE))
        model.eval()

        if not cfg.SPOTCHECK_PATH.exists():
            print(f"Spotcheck file missing at {cfg.SPOTCHECK_PATH}. Skipping spotcheck update.")
            return

        self._update_spotcheck_file(model, val_tf)

    def _update_spotcheck_file(self, model, val_tf):
        """
        Runs sliding-window inference over Nairobi imagery and writes predicted
        occupancy back into the spotcheck CSV. Mirrors the tail end of the
        original main() function exactly (facility-name matching, fallback to
        first image, occupancy fraction calculation, and CSV write-back).
        """
        cfg = self.config
        spotcheck_df = pd.read_csv(cfg.SPOTCHECK_PATH)

        if cfg.NAIROBI_IMAGERY_DIR.exists():
            # Find all valid image files inside NAIROBI_IMAGERY_DIR
            valid_exts = {".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG"}
            nairobi_images = [f for f in cfg.NAIROBI_IMAGERY_DIR.rglob("*") if f.is_file() and f.suffix in valid_exts]

            if not nairobi_images:
                print(f"No image files found in {cfg.NAIROBI_IMAGERY_DIR}")
                return

            print(
                f"Found {len(nairobi_images)} image(s) in {cfg.NAIROBI_IMAGERY_DIR.name}: {[i.name for i in nairobi_images]}")

            updated_rows = 0

            for idx, row in spotcheck_df.iterrows():
                facility = str(row["facility_name"]).strip()
                capacity = float(row.get("capacity", 100))

                img_candidate = None

                # 1. Exact or partial string match on facility name
                for img_f in nairobi_images:
                    if facility.lower() in img_f.name.lower():
                        img_candidate = img_f
                        break

                # 2. Fallback: use the first image in the folder if specific facility match isn't present
                if img_candidate is None:
                    img_candidate = nairobi_images[0]

                if img_candidate and img_candidate.is_file():
                    predicted_cars = SlidingWindowCounter.count_cars_in_large_image(
                        model, img_candidate, val_tf, cfg.DEVICE,
                        cfg.IMAGE_SIZE, cfg.PATCH_STRIDE, cfg.BATCH_SIZE
                    )
                    occupancy_frac = min(max(predicted_cars / capacity, 0.0), 1.0)

                    spotcheck_df.at[idx, "ground_truth_occupancy"] = round(predicted_cars, 1)
                    spotcheck_df.at[idx, "observed_occupancy_fraction"] = round(occupancy_frac, 4)
                    updated_rows += 1
                    print(
                        f"{facility} (Using image: {img_candidate.name}): Counted {predicted_cars:.1f} cars / {capacity:.0f} capacity -> Fraction: {occupancy_frac:.2%}")

            if updated_rows > 0:
                spotcheck_df.to_csv(cfg.SPOTCHECK_PATH, index=False)
                print(f"\nSuccessfully updated {updated_rows} rows in {cfg.SPOTCHECK_PATH}")
        else:
            print(f"Nairobi imagery folder {cfg.NAIROBI_IMAGERY_DIR} not found.")


ParkWiseModel1Pipeline

What it does: Runs the entire workflow in order, it loads the CSV, inspects and cleans columns, splits data by location, so the test set is a location the model never saw during training, builds the datasets with class-balanced sampling so as to to fix the "68% of patches have zero cars" imbalance, builds the model, trains it for the configured number of epochs while tracking MAE each epoch and saving the best checkpoint, then runs the trained model over real Nairobi imagery and writes the predicted occupancy back into the spotcheck CSV, so as to produce a trained, saved model file (.pt), console logs of training and test MAE per epoch, and an updated nairobi_parking_spotcheck.csv with real predicted occupancy values. This is the actual "run everything end-to-end" class. Everything above it is a building block.

In [8]:
if __name__ == "__main__":
    config = ParkWiseConfig()
    pipeline = ParkWiseModel1Pipeline(config)
    pipeline.run()

Using compute device: cpu
MODEL 1: CNN OVERHEAD CAR COUNTING PIPELINE
✓ Loaded 32773 records from object_count.csv.

--- CSV Structure Inspection ---
Columns available: ['Folder_Name', 'File_Name', 'Neg_Count', 'Other_Count', 'Pickup_Count', 'Sedan_Count', 'Unknown_Count']
Sample Row 0:
  - Folder_Name: Columbus_CSUAV_AFRL
  - File_Name: Columbus_EO_Run01_s2_301_15_00_31.99319028-Oct-2007_11-00-31.993_Frame_1.0.0.jpg
  - Neg_Count: 0
  - Other_Count: 2
  - Pickup_Count: 0
  - Sedan_Count: 1
  - Unknown_Count: 0
--------------------------------

✓ Summed vehicle columns ['Other_Count', 'Pickup_Count', 'Sedan_Count', 'Unknown_Count'] -> Target Column: 'Total_Car_Count'
Mapped Filename Column: 'File_Name' | Folder Column: 'Folder_Name'
✓ Split by location 'Potsdam' -> Train: 32136 | Test: 637
Building file map from dataset directory...
✓ Indexed 77492 total image files under DetectionPatches_512x512_ALL
Building file map from dataset directory...
✓ Indexed 77492 total image files under De

Epoch 1/15: 100%|██████████| 1005/1005 [11:44<00:00,  1.43it/s]


Epoch 01 | Train MAE: 2.562 | Test MAE: 2.912
  ✓ Saved best model checkpoint to /home/nia/Downloads/parkwise/model1_artifacts/parkwise_model1_resnet18.pt (MAE: 2.912)


Epoch 2/15: 100%|██████████| 1005/1005 [10:16<00:00,  1.63it/s]


Epoch 02 | Train MAE: 1.743 | Test MAE: 4.872


...

Epoch 13 | Train MAE: 0.875 | Test MAE: 2.316


Epoch 14/15: 100%|██████████| 1005/1005 [10:23<00:00,  1.61it/s]


Epoch 14 | Train MAE: 0.853 | Test MAE: 1.986


Epoch 15/15: 100%|██████████| 1005/1005 [10:24<00:00,  1.61it/s]


Epoch 15 | Train MAE: 0.847 | Test MAE: 2.011

Training Complete. Target Metric (MAE <= 2.5): Final Best MAE = 1.884

STEP 7: RUNNING INFERENCE ON NAIROBI PARKING LOTS
Found 95 image(s) in Images: ['WhatsApp Image 2026-08-28 at 1.18.17 PM (2).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.08 PM (1).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.13 PM (3).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.04 PM (3).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.19 PM (1).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.13 PM (2).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.06 PM (3).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.01 PM (1).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.09 PM.jpeg', 'WhatsApp Image 2026-08-28 at 1.18.18 PM (2).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.11 PM.jpeg', 'WhatsApp Image 2026-08-28 at 1.18.18 PM (3).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.01 PM (4).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.16 PM (2).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.13 PM (1).jpeg', 'WhatsApp Image 2026-08-28 at 

Entry Point

What it does: Creates a ParkWiseConfig, hands it to a ParkWiseModel1Pipeline, and calls .run(). What it produces: Triggers the entire pipeline when you run python parkwise_model1_oop.py from the command line. so as to execute

### MODEL 1B

apply_cnn_to_nairobi.py

Applies the trained Model 1 ResNet-18 car counting checkpoint to real Nairobi
parking lot images, slicing each image into padded patches, predicting a car
count per patch, and summing to an overall occupancy estimate per facility.


In [24]:
#IMPORTS
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torchvision import models, transforms

Imports

What it does: Pulls in pathlib for file paths, numpy for summing prediction arrays, PIL for opening images, and torch/torchvision for loading the model and preprocessing images.

In [26]:
#CONFIGURATION
class NairobiInferenceConfig:
    """
    Holds every constant the Nairobi inference script needs: where the trained
    Model 1 checkpoint lives, where the Nairobi imagery lives, which facility
    is currently being evaluated and its known capacity, the patch size used
    for slicing, the inference batch size, and the compute device.
    Identical values to the original script's "CONFIGURATION" section.
    """

    def __init__(self):
        self.MODEL_PATH = Path("/home/nia/Downloads/parkwise/parkwise_cowc_model/best_resnet18_cowc.pth")
        self.NAIROBI_IMAGES_DIR = Path("/home/nia/Downloads/parkwise/Images")

        self.FACILITY_NAME = "CBD_Holy_Family_Basement"  # Match name in nairobi_parking_master_dataset.csv
        self.PARKING_CAPACITY = 100

        self.PATCH_SIZE = 512
        self.BATCH_SIZE = 16  # Process patches in parallel batches
        self.DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NairobiInferenceConfig

What it does: Defines where the trained Model 1 checkpoint file lives, where the folder of Nairobi parking lot photos lives, which facility is currently being evaluated (by name, matching the master dataset) along with its known parking capacity, the patch size used to slice images (512px), the inference batch size, and which compute device to use. What it produces: A single config object every other class reads from. Why it's necessary: This script is meant to be re-run once per facility hence centralizing the facility name and capacity here means you only edit one place when you move on to the next parking lot, instead of hunting through inference code

In [27]:
#MODEL SETUP
class CarCounterModelLoader:
    """
    Wraps the original `load_car_counter_model` function.
    Rebuilds the same ResNet-18-with-regression-head architecture used during
    training (no ImageNet weights this time — the trained checkpoint provides
    them), loads the saved state dict from disk, and puts the model in eval
    mode on the target device. Logic unchanged from the original script.
    """

    @staticmethod
    def load_car_counter_model(model_path: Path, device: torch.device) -> nn.Module:
        """Loads ResNet-18 modified for continuous car count regression."""
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, 1)

        if model_path.exists():
            model.load_state_dict(torch.load(model_path, map_location=device))
            print(f"[+] Loaded model weights successfully from {model_path.name}")
        else:
            raise FileNotFoundError(f"Model checkpoint not found at: {model_path}")

        model = model.to(device)
        model.eval()
        return model


CarCounterModelLoader

What it does: Rebuilds the exact same ResNet-18 architecture (with its final layer swapped for a single-number regression output) that was used during training, but this time with no ImageNet pretrained weights, since the trained checkpoint will supply all the weights instead. Loads that checkpoint from disk into the architecture, moves it to the target device, and switches it to evaluation mode. What it produces: A trained nn.Module sitting in eval mode. Why it's necessary: PyTorch only saves the learned numbers (weights), not the architecture itself so the architecture has to be rebuilt in code before those weights can be loaded into it. Eval mode matters because it turns off training-only behaviors (like dropout) that would otherwise make predictions inconsistent.

In [28]:
#IMAGE PATCHING UTILS
class PatchExtractor:
    """
    Wraps the original `extract_padded_patches` function.
    Slices a full-resolution image into a grid of fixed-size patches. Any
    patch at the right/bottom edge that would be smaller than the target
    patch size gets zero-padded up to the full square instead of being
    resized/stretched, so its aspect ratio and scale stay correct.
    Logic unchanged from the original script.
    """

    @staticmethod
    def extract_padded_patches(img: Image.Image, patch_size: int):
        """
        Slices an image into uniform patches.
        Pads edge patches with zero-padding to prevent geometric distortion during resize.
        """
        width, height = img.size
        patches = []
        coords = []

        for y in range(0, height, patch_size):
            for x in range(0, width, patch_size):
                box = (x, y, min(x + patch_size, width), min(y + patch_size, height))
                cropped = img.crop(box)

                # Pad edge patch to square if necessary
                if cropped.size != (patch_size, patch_size):
                    padded = Image.new("RGB", (patch_size, patch_size), (0, 0, 0))
                    padded.paste(cropped, (0, 0))
                    cropped = padded

                patches.append(cropped)
                coords.append((x, y))
        return patches, coords

PatchExtractor

What it does: Cuts a full-size Nairobi parking lot photo into a grid of fixed-size (512×512) patches. Any patch that falls at the right or bottom edge of the image and would come out smaller than 512×512 gets padded with black pixels up to full size, rather than being stretched to fit. What it produces: A list of uniformly sized image patches plus their (x, y) coordinates in the original image. Why it's necessary: The trained model expects a consistent input size. Stretching an undersized edge patch to fit would distort car shapes and throw off predictions right at the image's border with zero-padding instead preserves the true scale of anything in that patch.

In [29]:
#MAIN PIPELINE
class NairobiInferencePipeline:
    """
    Wraps the original `main()` function.
    Loads the trained model, defines the same preprocessing transform used at
    training time, discovers every valid image in the Nairobi images
    directory, then for each image: slices it into padded patches, runs
    batched inference to predict a car count per patch, clips negative
    predictions to zero, sums them into a total car count, and converts that
    into an occupancy fraction/percentage against the configured capacity.
    Prints a per-file summary. Logic unchanged from the original script.
    """

    def __init__(self, config: NairobiInferenceConfig):
        self.config = config
        self.model = None
        self.transform = None

    def run(self):
        cfg = self.config

        # 1. Load Model
        model = CarCounterModelLoader.load_car_counter_model(cfg.MODEL_PATH, cfg.DEVICE)
        self.model = model

        # 2. Define Image Transformations
        transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])
        self.transform = transform

        # 3. Discover all valid image files in the directory
        valid_extensions = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
        image_paths = [
            p for p in cfg.NAIROBI_IMAGES_DIR.iterdir()
            if p.suffix.lower() in valid_extensions
        ]

        if not image_paths:
            raise FileNotFoundError(f"No valid image files found in {cfg.NAIROBI_IMAGES_DIR}")

        print(f"\n[+] Found {len(image_paths)} image(s) to process in {cfg.NAIROBI_IMAGES_DIR.name}\n")
        print("=" * 70)

        # 4. Process each image individually
        for img_path in image_paths:
            self._process_image(img_path)

    def _process_image(self, img_path: Path):
        """
        Runs patch extraction + batched inference + occupancy calculation for
        a single image, and prints its summary. Mirrors the body of the
        original script's per-image loop exactly.
        """
        cfg = self.config
        model = self.model
        transform = self.transform

        raw_image = Image.open(img_path).convert("RGB")
        patches, coordinates = PatchExtractor.extract_padded_patches(raw_image, cfg.PATCH_SIZE)

        all_counts = []

        with torch.no_grad():
            for i in range(0, len(patches), cfg.BATCH_SIZE):
                batch_patches = patches[i: i + cfg.BATCH_SIZE]
                tensors = torch.stack([transform(p) for p in batch_patches]).to(cfg.DEVICE)

                # Predict continuous car count per patch
                outputs = model(tensors).squeeze(-1)
                outputs = torch.clamp(outputs, min=0.0)
                all_counts.extend(outputs.cpu().numpy().tolist())

        # 5. Compute overall lot metrics
        total_cars_detected = float(np.sum(all_counts))
        ground_truth_occupancy_fraction = total_cars_detected / cfg.PARKING_CAPACITY
        occupancy_percentage = np.clip(ground_truth_occupancy_fraction * 100, 0.0, 100.0)

        # Output individual file summary
        print(f"File: {img_path.name}")
        print(f"  ├─ Patches Evaluated : {len(patches)}")
        print(f"  ├─ Estimated Cars    : {total_cars_detected:.2f}")
        print(f"  ├─ Occupancy Rate    : {occupancy_percentage:.2f}%")
        print(f"  └─ Ground Truth Frac : {ground_truth_occupancy_fraction:.4f}")
        print("-" * 70)



NairobiInferencePipeline

What it does: Loads the trained model, sets up the same resize preprocessing used during training, scans the Nairobi images folder for valid image files, and then for each image, slices it into padded patches, runs them through the model in batches, clips any negative predictions to zero, sums all patch-level predictions into one total car count for the whole lot, and converts that into an occupancy fraction and percentage against the facility's known capacity. Prints a summary per image (patches evaluated, estimated cars, occupancy rate, ground-truth fraction). What it produces: A printed per-facility occupancy estimate for every image found. Why it's necessary: This is the step that actually turns a raw aerial photo into the number ParkWise needs — a real occupancy percentage for a real Nairobi facility by using the model trained back in Model 1. It's also what feeds ground_truth_occupancy values into the spotcheck file that Model 2 depends on for calibration.

In [31]:
#ENTRY POINT (verbatim, including the original's trailing fragment)
if __name__ == "__main__":
    config = NairobiInferenceConfig()
    pipeline = NairobiInferencePipeline(config)
    pipeline.run()

    import joblib
import pandas as pd
from datetime import datetime

# Define destination filenames
MODEL_FILE = 'parkwise_occupancy_model.joblib'
METADATA_FILE = 'parkwise_model_metadata.joblib'

[+] Loaded model weights successfully from best_resnet18_cowc.pth

[+] Found 95 image(s) to process in Images

File: WhatsApp Image 2026-08-28 at 1.18.17 PM (2).jpeg
  ├─ Patches Evaluated : 4
  ├─ Estimated Cars    : 7.23
  ├─ Occupancy Rate    : 7.23%
  └─ Ground Truth Frac : 0.0723
----------------------------------------------------------------------
File: WhatsApp Image 2026-08-28 at 1.18.08 PM (1).jpeg
  ├─ Patches Evaluated : 4
  ├─ Estimated Cars    : 5.19
  ├─ Occupancy Rate    : 5.19%
  └─ Ground Truth Frac : 0.0519
----------------------------------------------------------------------
File: WhatsApp Image 2026-08-28 at 1.18.13 PM (3).jpeg
  ├─ Patches Evaluated : 4
  ├─ Estimated Cars    : 6.38
  ├─ Occupancy Rate    : 6.38%
  └─ Ground Truth Frac : 0.0638
----------------------------------------------------------------------
File: WhatsApp Image 2026-08-28 at 1.18.04 PM (3).jpeg
  ├─ Patches Evaluated : 4
  ├─ Estimated Cars    : 2.41
  ├─ Occupancy Rate    : 2.41%
  └─ Gro

Entry Point (and the original's trailing fragment)

What it does: Creates a NairobiInferenceConfig, hands it to a NairobiInferencePipeline, and calls .run() which triggers everything above. Then, the original script also had a dangling piece of code: an import joblib still inside the if __name__ block, followed by unindented import pandas, from datetime import datetime, and two unused filename constants (MODEL_FILE, METADATA_FILE). This class runs the whole inference pipeline.

### MODEL 2

ParkWise Model 2: Parking Pressure Predictor (Tuned Gradient Boosting Regressor)

### Description:
Trains a gradient boosting regression model to predict how busy a Nairobi parking facility will be (a 0–100 "parking pressure score") at a given date and time, using traffic congestion, time-of-day/day-of-week patterns, holidays, and recent occupancy history as inputs. Hyperparameters are tuned via time-aware cross-validation, and the model is benchmarked against a simple historical-average baseline before being saved for the backend to serve predictions.

In [9]:
import json
from pathlib import Path

import holidays
import joblib
import numpy as np
import pandas as pd
from scipy.stats import randint, uniform
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit


Imports

What it does: Pulls in pathlib for file handling, holidays for the Kenya public-holiday calendar, joblib for saving the trained model, numpy, pandas for data wrangling, scipy.stats for defining hyperparameter search distributions, and sklearn pieces for the model itself, its tuning, its evaluation metrics, and permutation importance.

In [10]:
#CONFIGURATION (original section "1. PATHS")
class ParkWiseModel2Config:
    """
    Holds every path used by the Model 2 pipeline: input CSVs (observations,
    spotcheck, master dataset, traffic log) and output artifact locations
    (trained model file, feature list JSON). Identical values to the original
    script's "1. PATHS" section.
    """

    def __init__(self):
        self.PROJECT = Path("/home/nia/Downloads/parkwise")

        self.TRAFFIC_LOG_PATH = self.PROJECT / "parkwise_final_maybe/nairobi_parking_traffic_log.csv"
        self.SPOTCHECK_PATH = self.PROJECT / "parkwise_final_maybe/nairobi_parking_spotcheck.csv"
        self.MASTER_DATASET_PATH = (
                self.PROJECT / "parkwise_final_maybe/nairobi_parking_master_dataset.csv"
        )
        self.EXPANDED_OBS_PATH = self.PROJECT / "nairobi_parking_expanded_observations.csv"

        self.ARTIFACT_DIR = self.PROJECT / "model2_artifacts"
        self.ARTIFACT_DIR.mkdir(exist_ok=True)

        self.MODEL_SAVE_PATH = self.ARTIFACT_DIR / "parkwise_model2_gbr.joblib"
        self.FEATURE_SAVE_PATH = self.ARTIFACT_DIR / "model2_feature_list.json"


ParkWiseModel2Config

What it does: Defines paths to every input file this model needs (traffic log, spotcheck, master dataset, expanded observations) and every output artifact location (saved model, feature list JSON), creating the artifact directory if it doesn't exist. What it produces: A single config object every other class reads from.

In [11]:
 #OBSERVATION LOADING (original section "2. LOAD OBSERVATIONS")
class ObservationLoader:
    """
    Loads the base observation dataset (preferring real spotcheck data if there
    are at least 50 rows, otherwise falling back to the synthetic expanded
    observations), then standardizes column names and builds the 0-100
    'parking_pressure_score' target column. Logic unchanged from the original
    script.
    """

    @staticmethod
    def load(config: ParkWiseModel2Config) -> pd.DataFrame:
        if config.SPOTCHECK_PATH.exists() and len(pd.read_csv(config.SPOTCHECK_PATH)) >= 50:
            obs_df = pd.read_csv(config.SPOTCHECK_PATH)
            print(f"✓ Using spotcheck dataset: {len(obs_df)} rows")
        elif config.EXPANDED_OBS_PATH.exists():
            obs_df = pd.read_csv(config.EXPANDED_OBS_PATH)
            print(f"✓ Using expanded observations: {len(obs_df)} rows")
        else:
            raise FileNotFoundError("No observation dataset available.")

        # Standardize columns
        if (
                "parking_pressure_score" in obs_df.columns
                and "observed_occupancy_fraction" not in obs_df.columns
        ):
            obs_df.rename(
                columns={"parking_pressure_score": "observed_occupancy_fraction"},
                inplace=True,
            )

        if "collected_at" in obs_df.columns and "timestamp" not in obs_df.columns:
            obs_df.rename(columns={"collected_at": "timestamp"}, inplace=True)

        if "observed_at" in obs_df.columns and "timestamp" not in obs_df.columns:
            obs_df.rename(columns={"observed_at": "timestamp"}, inplace=True)

        obs_df["timestamp"] = pd.to_datetime(obs_df["timestamp"], errors="coerce")
        obs_df = (
            obs_df.dropna(subset=["timestamp"])
            .sort_values("timestamp")
            .reset_index(drop=True)
        )

        # Target: 0 to 100 percentage score
        obs_df["parking_pressure_score"] = (
                obs_df["observed_occupancy_fraction"].astype(float).clip(0, 1) * 100
        )

        # Categorical Facility Names
        if "facility_name" not in obs_df.columns:
            obs_df["facility_name"] = "CBD_Default"

        return obs_df

ObservationLoader

Picks which dataset to train on, real spotcheck data if there are at least 50 rows, otherwise the synthetic expanded observations as a fallback. Then it standardizes inconsistent column names (collected_at timestamp, the mislabeled parking_pressure_score → observed_occupancy_fraction), drops rows with unparseable timestamps, and builds the real target column: parking_pressure_score on a 0–100 scale.

What it produces: A cleaned dataframe with a valid timestamp and a properly named 0–100 target column. Why it's necessary: The raw files have inconsistent naming across different data collection stages, and the model literally cannot train without a numeric target it can predict. This step guarantees that no matter which source file was used, the rest of the pipeline sees the same clean shape.

In [12]:
#METADATA & TRAFFIC MERGING (original section "3. MERGE FACILITY METADATA & TRAFFIC")
class MetadataMerger:
    """
    Merges facility metadata (zone, base rate, capacity) from the master
    dataset onto the observations by facility_name, extracts calendar fields
    (date/hour/dayofweek/month) from the timestamp, and merges in the traffic
    delay index — either matched by exact (date, hour) if the traffic log spans
    multiple dates, or by hour-only average profile if it only covers one date.
    Logic unchanged from the original script.
    """

    @staticmethod
    def merge(obs_df: pd.DataFrame, config: ParkWiseModel2Config) -> pd.DataFrame:
        merged_df = obs_df.copy()

        if config.MASTER_DATASET_PATH.exists():
            master_df = pd.read_csv(config.MASTER_DATASET_PATH)
            meta_cols = [
                c
                for c in ["facility_name", "zone", "base_rate_kes", "capacity"]
                if c in master_df.columns
            ]
            if "facility_name" in meta_cols:
                master_subset = master_df[meta_cols].drop_duplicates(
                    subset=["facility_name"]
                )
                merged_df = merged_df.merge(master_subset, on="facility_name", how="left")
                print("Facility metadata merged.")

        # Time extractors
        merged_df["date"] = merged_df["timestamp"].dt.date
        merged_df["hour"] = merged_df["timestamp"].dt.hour
        merged_df["dayofweek"] = merged_df["timestamp"].dt.dayofweek
        merged_df["month"] = merged_df["timestamp"].dt.month

        # Traffic Merge (Fallback to hourly pattern if single-date logs present)
        if config.TRAFFIC_LOG_PATH.exists():
            t_df = pd.read_csv(config.TRAFFIC_LOG_PATH)
            if "collected_at" in t_df.columns:
                t_df.rename(columns={"collected_at": "timestamp"}, inplace=True)
            t_df["timestamp"] = pd.to_datetime(t_df["timestamp"], errors="coerce")
            t_df["date"] = t_df["timestamp"].dt.date
            t_df["hour"] = t_df["timestamp"].dt.hour

            if "traffic_delay_index" in t_df.columns:
                if t_df["date"].nunique() > 1:
                    t_profile = (
                        t_df.groupby(["date", "hour"])["traffic_delay_index"]
                        .mean()
                        .reset_index()
                    )
                    merged_df = merged_df.merge(t_profile, on=["date", "hour"], how="left")
                else:
                    t_profile = (
                        t_df.groupby("hour")["traffic_delay_index"].mean().reset_index()
                    )
                    merged_df = merged_df.merge(t_profile, on="hour", how="left")
                    print("Traffic log has 1 date: Using mean hourly profile fallback.")

        return merged_df

MetadataMerger

What it does: Joins in facility-level metadata (zone, base hourly rate, capacity) from the master dataset by facility name. Extracts date, hour day-of-week, month from the timestamp. Merges in the traffic delay index, matched exactly by date+hour if the traffic log spans multiple days, or averaged by hour-only if the log only covers a single day. What it produces: A dataframe enriched with facility attributes, calendar fields, and traffic signal. Why it's necessary: The model needs to know which facility, when, and how congested the roads were, none of that lives in the observation file alone. The single-day fallback matters specifically because if Person 1's traffic log only covers one day, matching by exact date would leave every other row with no traffic signal at all; falling back to an hourly average keeps the feature usable instead of empty.

In [13]:
#LAG FEATURE CREATION (original section "4. ROBUST LAG FEATURE CREATION WITH TIME GAUGING")
class LagFeatureBuilder:
    """
    Builds time-aware "previous occupancy" and "rolling 3-observation average"
    features per facility. A previous observation is only used as a valid lag
    if it happened within 3 hours of the current one (via time_diff_hours);
    otherwise it's treated as missing and later backfilled with that facility's
    median pressure score. Logic unchanged from the original script.
    """

    @staticmethod
    def build(merged_df: pd.DataFrame) -> pd.DataFrame:
        print("\n" + "=" * 70)
        print("BUILDING TIME-AWARE FACILITY OCCUPANCY LAGS")
        print("=" * 70)

        merged_df = merged_df.sort_values(["facility_name", "timestamp"]).reset_index(
            drop=True
        )

        merged_df["time_diff_hours"] = (
                merged_df.groupby("facility_name")["timestamp"].diff().dt.total_seconds()
                / 3600.0
        )

        facility_group = merged_df.groupby("facility_name")["parking_pressure_score"]
        merged_df["raw_prev_occ"] = facility_group.shift(1)

        merged_df["previous_occupancy"] = np.where(
            merged_df["time_diff_hours"] <= 3.0, merged_df["raw_prev_occ"], np.nan
        )

        merged_df["occupancy_rolling_3"] = merged_df.groupby("facility_name")[
            "previous_occupancy"
        ].transform(lambda x: x.rolling(3, min_periods=1).mean())

        facility_medians = merged_df.groupby("facility_name")[
            "parking_pressure_score"
        ].transform("median")
        merged_df["previous_occupancy"] = merged_df["previous_occupancy"].fillna(
            facility_medians
        )
        merged_df["occupancy_rolling_3"] = merged_df["occupancy_rolling_3"].fillna(
            facility_medians
        )

        return merged_df


LagFeatureBuilder

What it does: For each facility, looks at the previous observation's pressure score and only counts it as valid history if it happened within 3 hours (otherwise treats it as missing). Builds a rolling 3 observation average per facility. Fills any remaining gaps with that facility's overall median pressure score. What it produces: Two new columns, previous_occupancy and occupancy_rolling_3 with no missing values. Why it's necessary: What was this spot's occupancy an hour ago is often one of the single strongest predictors . The 3-hour cutoff stops the model from treating a reading from three days ago as if it were current, and the median fallback ensures a facility with sparse history still gets a sensible starting value instead of a blank.

In [14]:
#CYCLIC/CALENDAR FEATURES (original section "5. CYCLIC AND CALENDAR FEATURES")
class CalendarFeatureEngineer:
    """
    Builds sine/cosine encodings of hour-of-day and day-of-week, weekend and
    peak-hour flags, and a Kenya public-holiday flag (via the `holidays`
    library). Also assembles the candidate feature list, conditionally adding
    traffic_delay_index and base_rate_kes (with median imputation) if those
    columns exist. Logic unchanged from the original script.
    """

    @staticmethod
    def build(merged_df: pd.DataFrame) -> tuple[pd.DataFrame, list]:
        hour = merged_df["hour"]
        day = merged_df["dayofweek"]

        merged_df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
        merged_df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
        merged_df["day_sin"] = np.sin(2 * np.pi * day / 7)
        merged_df["day_cos"] = np.cos(2 * np.pi * day / 7)
        merged_df["is_weekend"] = day.isin([5, 6]).astype(int)
        merged_df["is_peak_hour"] = hour.isin([7, 8, 9, 16, 17, 18]).astype(int)

        ke_holidays = holidays.Kenya()
        merged_df["is_public_holiday"] = merged_df["date"].apply(
            lambda d: int(d in ke_holidays)
        )

        merged_df["facility_name"] = merged_df["facility_name"].astype("category")

        candidate_features = [
            "facility_name",
            "hour_sin",
            "hour_cos",
            "day_sin",
            "day_cos",
            "is_weekend",
            "is_peak_hour",
            "is_public_holiday",
            "previous_occupancy",
            "occupancy_rolling_3",
        ]

        if "traffic_delay_index" in merged_df.columns:
            merged_df["traffic_delay_index"] = (
                pd.to_numeric(merged_df["traffic_delay_index"], errors="coerce").fillna(
                    merged_df["traffic_delay_index"].median()
                )
            )
            candidate_features.append("traffic_delay_index")

        if "base_rate_kes" in merged_df.columns:
            merged_df["base_rate_kes"] = (
                pd.to_numeric(merged_df["base_rate_kes"], errors="coerce").fillna(
                    merged_df["base_rate_kes"].median()
                )
            )
            candidate_features.append("base_rate_kes")

        return merged_df, candidate_features


CalendarFeatureEngineer

What it does: Converts hour-of-day and day-of-week into sine/cosine pairs (so hour 23 and hour 0 register as close together instead of far apart), flags weekends and peak commuting hours, flags Kenyan public holidays using the holidays library, and assembles the final candidate feature list  adding traffic_delay_index and base_rate_kes only if those columns actually exist, with missing values filled by the median. What it produces: The full engineered feature set plus a candidate_features list naming every column the model might use. Why it's necessary: Raw hourday numbers mislead a model into thinking 11pm and midnight are unrelated, cyclic encoding fixes that. Holiday and peak-hour flags capture real-world demand spikes a model couldn't infer from time alone. Making traffic rate columns conditional means the pipeline doesn't crash if one of those input files is missing a column.

In [15]:
#CHRONOLOGICAL SPLIT (original section "6. CHRONOLOGICAL SPLIT (80/20)")
class ChronologicalSplitter:
    """
    Sorts all rows by timestamp and takes the first 80% as training data and
    the last 20% as test data (never shuffled, to avoid leaking future
    observations into training). Then filters the candidate feature list down
    to "active" features — any non-facility_name feature with fewer than 2
    unique values in the training set is dropped as uninformative.
    Logic unchanged from the original script.
    """

    @staticmethod
    def split(merged_df: pd.DataFrame, candidate_features: list):
        merged_df = merged_df.sort_values("timestamp").reset_index(drop=True)
        split_idx = int(len(merged_df) * 0.80)

        train_df = merged_df.iloc[:split_idx].copy()
        test_df = merged_df.iloc[split_idx:].copy()

        active_features = []
        categorical_features = []

        for c in candidate_features:
            if c == "facility_name":
                active_features.append(c)
                categorical_features.append(c)
                continue

            if train_df[c].nunique(dropna=True) >= 2:
                active_features.append(c)
            else:
                print(
                    f"Dropped '{c}': Low training variance ({train_df[c].nunique()} unique"
                    " value)."
                )

        X_train = train_df[active_features]
        X_test = test_df[active_features]
        y_train = train_df["parking_pressure_score"].astype(float)
        y_test = test_df["parking_pressure_score"].astype(float)

        print("\n" + "=" * 70)
        print("DATA SPLIT")
        print("=" * 70)
        print(f"Train Rows: {len(X_train)} | Test Rows: {len(X_test)}")
        print(f"Active Features ({len(active_features)}): {active_features}")

        return train_df, test_df, X_train, X_test, y_train, y_test, active_features, categorical_features


ChronologicalSplitter

What it does: Sorts every row by timestamp, takes the first 80% as training data and the last 20% as test data. Then checks each candidate feature and drops any (other than facility_name) that has fewer than 2 unique values in the training set, since a constant column can't teach the model anything. What it produces: X_train, X_test, y_train, y_test, and the final active_features list actually used for training. Why it's necessary: Shuffling before splitting would let the model "see the future" during training and test on the past — producing misleadingly great metrics that fall apart in real use. The variance check prevents wasting model capacity on a feature that's the same value for every row which would just add noise.

In [16]:
#HISTORICAL BASELINE (original section "7. HISTORICAL BASELINE COMPARISON")
class HistoricalBaseline:
    """
    Computes the simplest possible prediction: for each (facility, hour,
    day-of-week) combination in the training set, what was the historical
    average parking pressure? Applies that average to the test set (falling
    back to the overall training mean where no historical match exists) and
    reports MAE/RMSE. This is the bar the real model must beat.
    Logic unchanged from the original script.
    """

    @staticmethod
    def evaluate(train_df: pd.DataFrame, test_df: pd.DataFrame, y_train: pd.Series, y_test: pd.Series):
        historical_avg = (
            train_df.groupby(["facility_name", "hour", "dayofweek"], observed=False)[
                "parking_pressure_score"
            ]
            .mean()
            .reset_index()
            .rename(columns={"parking_pressure_score": "baseline_prediction"})
        )

        test_baseline = test_df.merge(
            historical_avg, on=["facility_name", "hour", "dayofweek"], how="left"
        )
        test_baseline["baseline_prediction"] = test_baseline[
            "baseline_prediction"
        ].fillna(y_train.mean())

        baseline_preds = test_baseline["baseline_prediction"].to_numpy()
        baseline_mae = mean_absolute_error(y_test, baseline_preds)
        baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_preds))

        print("\n" + "=" * 70)
        print("HISTORICAL BASELINE PERFORMANCE")
        print("=" * 70)
        print(f"Baseline MAE:  {baseline_mae:.3f}")
        print(f"Baseline RMSE: {baseline_rmse:.3f}")

        return baseline_mae, baseline_rmse


HistoricalBaseline

What it does: Computes the simplest possible prediction, the historical average pressure score for each (facility, hour, day-of-week) combination seen in training and applies it to the test set, falling back to the overall training mean when no exact historical match exists. Reports its MAE and RMSE. What it produces: baseline_mae and baseline_rmse two numbers the real model must beat to be worth using. Why it's necessary: Without this, you have no way to know if the gradient boosting model actually learned anything useful or is just an expensive way to reproduce what a simple lookup table already knew.

In [17]:
#MODEL TRAINING (original section "8. TRAIN MODEL 2 (HYPERPARAMETER TUNING VIA RANDOMIZEDSEARCHCV)")
class GBRTrainer:
    """
    Defines the hyperparameter search space for HistGradientBoostingRegressor,
    then runs RandomizedSearchCV (25 iterations) scored on negative MAE, using
    TimeSeriesSplit (5 folds) as the cross-validation strategy so that no fold
    ever validates on data that occurred before its training data.
    Logic unchanged from the original script.
    """

    @staticmethod
    def train(X_train, y_train, categorical_features: list):
        param_distributions = {
            "max_iter": randint(150, 450),
            "learning_rate": uniform(0.01, 0.08),
            "max_depth": randint(3, 8),
            "min_samples_leaf": randint(15, 60),
            "l2_regularization": uniform(0.5, 5.0),
            "max_leaf_nodes": randint(15, 63),
        }

        base_model = HistGradientBoostingRegressor(
            categorical_features=categorical_features, random_state=42
        )

        tscv = TimeSeriesSplit(n_splits=5)

        search = RandomizedSearchCV(
            estimator=base_model,
            param_distributions=param_distributions,
            n_iter=25,
            scoring="neg_mean_absolute_error",
            cv=tscv,
            random_state=42,
            n_jobs=-1,
            verbose=1,
        )

        print("\n" + "=" * 70)
        print("EXECUTING RANDOMIZED SEARCH CV ACROSS TIME SERIES SPLITS")
        print("=" * 70)

        search.fit(X_train, y_train)

        m2_model = search.best_estimator_

        print("\n✓ Tuning Complete!")
        print(f"Best CV Mean MAE: {-search.best_score_:.3f}")
        print("Best Hyperparameters:")
        for param, val in search.best_params_.items():
            print(f"  • {param}: {val}")

        return m2_model, search



GBRTrainer

What it does: Defines ranges for six hyperparameters (tree count, learning rate, depth, minimum leaf size, L2 regularization, max leaf nodes), then runs RandomizedSearchCV trying 25 random combinations, each evaluated using TimeSeriesSplit 5 folds that always validate on data occurring after their training fold. What it produces: m2_model  the best-performing trained HistGradientBoostingRegressor found across all 25 trials  plus the full search object. Why it's necessary: Picking hyperparameters by hand is guesswork; a randomized search over a defined range finds a genuinely better tuned model with much less manual trial and error. TimeSeriesSplit matters because a normal random cross-validation would (again) leak future data into training during the search itself exactly the mistake Cell 7 avoids at the final split

In [18]:
#EVALUATION (original section "9. EVALUATION")
class ModelEvaluator:
    """
    Predicts on the held-out test set (clipping predictions to the valid 0-100
    range), computes MAE/RMSE, and compares directly against the historical
    baseline's MAE to report the improvement (or regression) in both absolute
    points and percentage terms. Logic unchanged from the original script.
    """

    @staticmethod
    def evaluate(m2_model, X_test, y_test, baseline_mae: float):
        m2_preds = np.clip(m2_model.predict(X_test), 0, 100)
        m2_mae = mean_absolute_error(y_test, m2_preds)
        m2_rmse = np.sqrt(mean_squared_error(y_test, m2_preds))
        mae_improvement = baseline_mae - m2_mae

        print("\n" + "=" * 70)
        print("MODEL 2: GRADIENT BOOSTING RESULTS")
        print("=" * 70)
        print(f"Model 2 MAE:  {m2_mae:.3f}")
        print(f"Model 2 RMSE: {m2_rmse:.3f}")

        if m2_mae < baseline_mae:
            imp_pct = (mae_improvement / baseline_mae) * 100
            print(
                f"✓ MAE Improvement over Baseline: {mae_improvement:.3f} points"
                f" ({imp_pct:.2f}%)"
            )
        else:
            print(
                f"Model 2 MAE is {abs(mae_improvement):.3f} points worse than"
                " baseline."
            )

        return m2_preds, m2_mae, m2_rmse

ModelEvaluator

What it does: Predicts on the untouched test set, clips predictions to the valid 0–100 range (since occupancy can't be negative or over 100%), computes MAE and RMSE, and compares directly against the Cell 8 baseline to report the improvement (or, honestly, the regression) in both raw points and percentage. What it produces: m2_preds, m2_mae, m2_rmse. Why it's necessary: This is the actual verdict on whether the tuned model earns its place over the simple baseline. Clipping predictions matters because a raw regressor has no builtin awareness that its output represents a percentage.

In [19]:
#PERMUTATION IMPORTANCE (original section "10. PERMUTATION IMPORTANCE")
class ImportanceAnalyzer:
    """
    Runs scikit-learn's permutation importance on the test set (5 repeats,
    scored on negative MAE) to measure how much each feature actually
    contributes to prediction accuracy, then prints features ranked from most
    to least important. Logic unchanged from the original script.
    """

    @staticmethod
    def analyze(m2_model, X_test, y_test, active_features: list):
        print("\n" + "=" * 70)
        print("PERMUTATION IMPORTANCE")
        print("=" * 70)

        perm = permutation_importance(
            m2_model,
            X_test,
            y_test,
            scoring="neg_mean_absolute_error",
            n_repeats=5,
            random_state=42,
            n_jobs=-1,
        )

        importance = pd.Series(
            perm.importances_mean, index=active_features
        ).sort_values(ascending=False)
        for feat, imp in importance.items():
            print(f"{feat:<25} {imp:>10.5f}")

        return importance


ImportanceAnalyzer

What it does: Runs permutation importance on the test set  repeatedly shuffling each feature's values one at a time and measuring how much the model's error gets worse then ranks every feature from most to least important. What it produces: A ranked importance series printed to console. Why it's necessary: This tells you and the team which inputs are actually driving predictions. It's the same spirit as the "sanity check" step in the original instructions — an honest look at what the model actually learned, not just how accurate it is overall.

In [20]:
#ARTIFACT SAVING (original section "11. SAVE ARTIFACTS")
class ArtifactSaver:
    """
    Persists the trained model to disk via joblib, and writes a JSON file
    listing exactly which features the model was trained on (and in what
    order) plus the target column name — so anyone serving predictions later
    knows the exact input contract. Logic unchanged from the original script.
    """

    @staticmethod
    def save(m2_model, active_features: list, config: ParkWiseModel2Config):
        joblib.dump(m2_model, config.MODEL_SAVE_PATH)
        with open(config.FEATURE_SAVE_PATH, "w") as f:
            json.dump(
                {"features": active_features, "target": "parking_pressure_score"},
                f,
                indent=4,
            )

        print("\n" + "=" * 70)
        print(f"✓ Model saved: {config.MODEL_SAVE_PATH}")
        print(f"✓ Features saved: {config.FEATURE_SAVE_PATH}")
        print("=" * 70)


ArtifactSaver

What it does: Saves the trained model to disk with joblib, and writes a JSON file listing exactly which features the model expects and in what order, alongside the target column name. What it produces: parkwise_model2_gbr.joblib and model2_feature_list.json on disk. Why it's necessary: Whoever serves this model in the backend (Person 5) needs to know the exact input contract which columns, in which order or predictions will silently be wrong. Saving this alongside the model instead of just documenting it separately means the contract can never drift out of sync with the actual trained model.

In [21]:
#PIPELINE ORCHESTRATOR
class ParkWiseModel2Pipeline:
    """
    Runs the entire Model 2 workflow in the same order as the original script:
    load observations -> merge metadata/traffic -> build lag features -> build
    calendar features -> chronological split -> historical baseline -> train
    tuned GBR -> evaluate -> permutation importance -> save artifacts.

    Every computation is identical to the original script's top-level flow;
    only the surrounding structure (a config object and delegation to the
    helper classes above) differs.
    """

    def __init__(self, config: ParkWiseModel2Config):
        self.config = config
        self.model = None
        self.active_features = None
        self.importance = None

    def run(self):
        cfg = self.config

        print("=" * 70)
        print("PARKWISE MODEL 2: OPTIMIZED PARKING PRESSURE PREDICTOR")
        print("=" * 70)

        obs_df = ObservationLoader.load(cfg)
        merged_df = MetadataMerger.merge(obs_df, cfg)
        merged_df = LagFeatureBuilder.build(merged_df)
        merged_df, candidate_features = CalendarFeatureEngineer.build(merged_df)

        (train_df, test_df, X_train, X_test, y_train, y_test,
         active_features, categorical_features) = ChronologicalSplitter.split(merged_df, candidate_features)

        baseline_mae, baseline_rmse = HistoricalBaseline.evaluate(train_df, test_df, y_train, y_test)

        m2_model, search = GBRTrainer.train(X_train, y_train, categorical_features)

        m2_preds, m2_mae, m2_rmse = ModelEvaluator.evaluate(m2_model, X_test, y_test, baseline_mae)

        importance = ImportanceAnalyzer.analyze(m2_model, X_test, y_test, active_features)

        ArtifactSaver.save(m2_model, active_features, cfg)

        self.model = m2_model
        self.active_features = active_features
        self.importance = importance


ParkWiseModel2Pipeline

What it does: Calls every class above in the correct order load, merge, lag features, calendar features, split, baseline, train, evaluate, importance, save exactly matching the original script's top-to-bottom flow. What it produces: A fully trained and saved model, plus every console log the original script produced along the way. Why it's necessary: This is the class that actually wires everything together and guarantees the steps run in the one order that makes the results valid (e.g., you can't evaluate before you split, can't split before you engineer features). Everything above it is a building block; this is what executes the full pipeline.

In [22]:
#ENTRY POINT
if __name__ == "__main__":
    config = ParkWiseModel2Config()
    pipeline = ParkWiseModel2Pipeline(config)
    pipeline.run()

PARKWISE MODEL 2: OPTIMIZED PARKING PRESSURE PREDICTOR
✓ Using expanded observations: 36621 rows
Facility metadata merged.
Traffic log has 1 date: Using mean hourly profile fallback.

BUILDING TIME-AWARE FACILITY OCCUPANCY LAGS
Dropped 'is_public_holiday': Low training variance (1 unique value).
Dropped 'traffic_delay_index': Low training variance (1 unique value).

DATA SPLIT
Train Rows: 29296 | Test Rows: 7325
Active Features (10): ['facility_name', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'is_weekend', 'is_peak_hour', 'previous_occupancy', 'occupancy_rolling_3', 'base_rate_kes']

HISTORICAL BASELINE PERFORMANCE
Baseline MAE:  10.814
Baseline RMSE: 13.701

EXECUTING RANDOMIZED SEARCH CV ACROSS TIME SERIES SPLITS
Fitting 5 folds for each of 25 candidates, totalling 125 fits

✓ Tuning Complete!
Best CV Mean MAE: 8.639
Best Hyperparameters:
  • l2_regularization: 2.729163764267956
  • learning_rate: 0.01799799326544023
  • max_depth: 5
  • max_iter: 237
  • max_leaf_nodes: 50
  • m

Entry Point

What it does: Creates a ParkWiseModel2Config, hands it to a ParkWiseModel2Pipeline, and calls .run(). What it produces: Triggers the entire Model 2 pipeline when you run python parkwise_model2_oop.py.